# Neural Network Quantization Visualization

This notebook shows the effects of quantization on the MobileNetV2 ONNX model. Inference is performed with ONNXRuntime with TensorRT execution provider. The original FP32 model inference speed and accuracy metrics are compared to its FP16 and INT8 quantizations. The feature map differences are visualized with plots.

## Requirements

Install the requirements with 

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import numpy as np
import time
import random
import os
import sys
from onnxruntime.quantization import CalibrationMethod
from onnxruntime.quantization.shape_inference import quant_pre_process
import onnxruntime as ort
from onnxruntime.quantization import CalibrationMethod, create_calibrator, write_calibration_table
from copy import deepcopy   
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import tempfile
from copy import deepcopy
import onnx

Since we are not installing TensorRT libraries system-wide, we must add them to `LD_LIBRARY_PATH` for this notebook session.

In [ ]:
venv_path = sys.prefix

nvidia_package_dirs = [
    "cublas/lib",
    "cudnn/lib",
    "cuda_runtime/lib",
    "cufft/lib",
    "curand/lib",
    "cusolver/lib",
    "cusparse/lib",
    "tensorrt_libs"
]

nvidia_libs = [f"{venv_path}/lib/python3.12/site-packages/nvidia/{d}" for d in nvidia_package_dirs]
nvidia_libs.append(f"{venv_path}/lib/python3.12/site-packages/tensorrt_libs")
current_ld_path = os.environ.get("LD_LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_libs) + (f":{current_ld_path}" if current_ld_path else "")

This notebook requires an NVIDIA GPU with compute capability 7.5 or newer. The cell below must outptut `device(type='cuda')`.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Dataset

We will use a subset of ImageNet-1k dataset for testing. Download it from here and extract the archive.

In [ ]:
BATCH_SIZE = 1
MODEL_NAME = "mobilenet_v2"
IMG_DIM = 224

In [ ]:
from PIL import Image
from torchvision.transforms import InterpolationMode

# Set the path to the imagenet1k directory
IMAGENET_ROOT = "imagenet1k"

# These values are commonly used
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Images must be converted to RGB, resized, normalized with ImageNet means and standard deviations, and converted to NCHW format.
def rgb_loader(path):
    with Image.open(path) as img:
        return img.convert("RGB") # RGB

imagenet_transform = transforms.Compose([
    transforms.Resize((IMG_DIM, IMG_DIM), interpolation=InterpolationMode.BILINEAR, antialias=True), # resize
    transforms.ToTensor(), # to NCHW
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), # normalize
])

full_dataset = torchvision.datasets.ImageFolder(
    root=IMAGENET_ROOT,
    loader=rgb_loader,
    transform=imagenet_transform
)
num_images = len(full_dataset)

print(f"Total ImageNet samples found: {num_images}")

## Load Model
First, we must download the Pytorch MobileNetV2 model from `torchvision` and convert it to ONNX.

In [ ]:
model = torchvision.models.mobilenet_v2(weights='DEFAULT')
model.to(device)
model.eval()

dummy_input = torch.randn(BATCH_SIZE, 3, IMG_DIM, IMG_DIM).to(device)

base_onnx_model_name = f"{MODEL_NAME}.onnx"

torch.onnx.export(
    model,
    dummy_input,
    base_onnx_model_name,
    export_params=True,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output']
)

In [ ]:
# Run shape inference on the model to get better quantization results.
preprocessed_onnx_model_name = f"{MODEL_NAME}_preproc.onnx"

In [ ]:

quant_pre_process(
    input_model=base_onnx_model_name, 
    output_model_path=preprocessed_onnx_model_name,
    skip_symbolic_shape=False
)

## Get INT8 Calibration Table
We will use 500 random images from the dataset for calibration, and 100 images for testing.

In [ ]:
NUM_CALIBRATION_IMAGES = 500
NUM_TEST_IMAGES = 5000

all_indices = list(range(num_images))
random.shuffle(all_indices)

calib_indices = all_indices[:NUM_CALIBRATION_IMAGES]
benchmark_indices = all_indices[NUM_CALIBRATION_IMAGES:NUM_CALIBRATION_IMAGES + NUM_TEST_IMAGES]

calibration_dataset = torch.utils.data.Subset(full_dataset, calib_indices)
benchmark_dataset = torch.utils.data.Subset(full_dataset, benchmark_indices)

calibration_loader = torch.utils.data.DataLoader(
    calibration_dataset,
    batch_size=1,
    shuffle=False,
    drop_last=False
)

benchmark_loader = torch.utils.data.DataLoader(
    benchmark_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=True
)

print(f"Calibration samples: {len(calibration_dataset)}")
print(f"Test samples: {len(benchmark_dataset)}")

In [ ]:
MODEL_FILE = preprocessed_onnx_model_name
TRT_CACHE_DIR = "trt_cache" 

In [ ]:

os.makedirs(TRT_CACHE_DIR, exist_ok=True)

# Create a simple data loader
class ImageNetCalibrator:
    def __init__(self, dataloader, num_batches=None, input_name="input"):
        self.source_loader = dataloader
        self.dataloader = iter(dataloader)
        self.num_batches = num_batches if num_batches is not None else len(dataloader)
        self.current_batch = 0
        self.input_name = input_name

    def get_next(self):
        if self.current_batch < self.num_batches:
            try:
                images, _ = next(self.dataloader)
                self.current_batch += 1
                return {self.input_name: images.numpy()}
            except StopIteration:
                return None
        return None

    def rewind(self):
        self.current_batch = 0
        self.dataloader = iter(self.source_loader)

# Create calibrator
calibrator = create_calibrator(
    MODEL_FILE,
    augmented_model_path="augmented_model.onnx", # this output model is not going to be used
    calibrate_method=CalibrationMethod.MinMax # Other options are Entropy and Percentile
)

# Create the data loader
imagenet_data_reader = ImageNetCalibrator(calibration_loader, num_batches=len(calibration_loader), input_name="input")

calibrator.set_execution_providers(["TensorrtExecutionProvider", "CUDAExecutionProvider", "CPUExecutionProvider"])

# Calculate the quantization scales and write the calibration table.
# Three files will be saved in the TRT_CACHE_DIR:
# calibration.cache
# calibration.flatbuffers
# calibration.json
# The 'calibration.flatbuffers' file will be passed to the TensorRT execution provider
calibrator.collect_data(data_reader=imagenet_data_reader)
write_calibration_table(calibrator.compute_data().data, dir=str(TRT_CACHE_DIR))

In [ ]:
def benchmark_speed(model_path, benchmark_dataloader, mode="FP32", iterations=None):
    use_fp16 = (mode == "FP16" or mode == "INT8")
    use_int8 = (mode == "INT8")

    trt_options = {
        'device_id': 0,
        'trt_fp16_enable': use_fp16,
        'trt_int8_enable': use_int8,
        'trt_engine_cache_enable': True,
        'trt_engine_cache_path': './trt_cache',
        'trt_timing_cache_enable': True,
        'trt_timing_cache_path': './trt_cache',
    }

    if use_int8:
        trt_options.update({
            'trt_int8_use_native_calibration_table': False,
            'trt_int8_calibration_table_name': 'calibration.flatbuffers',
        })

    # We must use TensorRT for this example, but still provide CUDA and CPU as backup providers.
    provider_list = ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
    options_list = [trt_options, {}, {}]

    sess_options = ort.SessionOptions()
    sess_options.add_session_config_entry('session.dynamic_block_base', '4')

    # The first time the session runs, TensorRT will generate the engine cache for the model. It may take a while, be patient!
    print("Creating session...")
    session = ort.InferenceSession(
        model_path,
        providers=provider_list,
        provider_options=options_list,
        sess_options=sess_options
    )
    print("Created session")

    input_name = session.get_inputs()[0].name

    warmup_steps = min(20, len(benchmark_dataloader))
    print(f"Warming up {mode}...")
    warmup_iter = iter(benchmark_dataloader)
    for _ in range(warmup_steps):
        try:
            images, _ = next(warmup_iter)
        except StopIteration:
            warmup_iter = iter(benchmark_dataloader)
            images, _ = next(warmup_iter)
        session.run(None, {input_name: images.numpy()})

    if iterations is None:
        iterations = len(benchmark_dataloader)

    latencies = []
    bench_iter = iter(benchmark_dataloader)
    print(f"Benchmarking {mode}...")
    for _ in range(iterations):
        try:
            images, _ = next(bench_iter)
        except StopIteration:
            bench_iter = iter(benchmark_dataloader)
            images, _ = next(bench_iter)

        start = time.perf_counter()
        session.run(None, {input_name: images.numpy()})
        latencies.append((time.perf_counter() - start) * 1000)

    mean_lat = np.mean(latencies)
    throughput = benchmark_dataloader.batch_size / (mean_lat / 1000)

    print(f"--- {mode} Results ---")
    print(f"Mean Latency:    {mean_lat:.4f} ms")
    print(f"Throughput:      {throughput:.2f} images/sec\n")

    return mean_lat


lat_32 = benchmark_speed(MODEL_FILE, benchmark_loader, mode="FP32")
lat_16 = benchmark_speed(MODEL_FILE, benchmark_loader, mode="FP16")
lat_08 = benchmark_speed(MODEL_FILE, benchmark_loader, mode="INT8")

print("Model Comparison:")
print(f"FP32: {lat_32:.4f} ms")
print(f"FP16: {lat_16:.4f} ms ({lat_32/lat_16:.2f}x speedup)")
print(f"INT8: {lat_08:.4f} ms ({lat_32/lat_08:.2f}x speedup)")

In [ ]:
def measure_classification_metrics_trt(model_path, benchmark_dataloader, mode="FP32", verify_preprocessing=True):
    use_fp16 = (mode == "FP16" or mode == "INT8")
    use_int8 = (mode == "INT8")

    trt_options = {
        'device_id': 0,
        'trt_fp16_enable': use_fp16,
        'trt_int8_enable': use_int8,
        'trt_engine_cache_enable': True,
        'trt_engine_cache_path': './trt_cache',
        'trt_timing_cache_enable': True,
        'trt_timing_cache_path': './trt_cache',
    }

    if use_int8:
        trt_options.update({
            'trt_int8_use_native_calibration_table': False,
            'trt_int8_calibration_table_name': 'calibration.flatbuffers',
        })

    provider_list = ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
    options_list = [trt_options, {}, {}]

    sess_options = ort.SessionOptions()
    sess_options.add_session_config_entry('session.dynamic_block_base', '4')

    print(f"Creating {mode} session for metrics...")
    session = ort.InferenceSession(
        model_path,
        providers=provider_list,
        provider_options=options_list,
        sess_options=sess_options
    )

    input_name = session.get_inputs()[0].name

    subset = benchmark_dataloader.dataset
    if not hasattr(subset, "dataset") or not hasattr(subset, "indices"):
        raise ValueError("Expected benchmark_dataloader.dataset to be torch.utils.data.Subset(ImageFolder)")

    base_dataset = subset.dataset
    subset_indices = subset.indices

    y_true = []
    y_pred = []

    cursor = 0
    checks_remaining = 3

    for images, _ in tqdm(benchmark_dataloader, desc=f"Evaluating {mode}"):
        # Enforce expected model input format: float32 NCHW, RGB normalized tensor
        if images.dtype != torch.float32:
            images = images.float()
        if images.ndim != 4 or images.shape[1:] != (3, IMG_DIM, IMG_DIM):
            raise ValueError(f"Expected NCHW=(N,3,{IMG_DIM},{IMG_DIM}), got {tuple(images.shape)}")

        batch_size = images.shape[0]
        batch_indices = subset_indices[cursor:cursor + batch_size]
        cursor += batch_size

        if verify_preprocessing and checks_remaining > 0:
            for local_i, ds_idx in enumerate(batch_indices):
                if checks_remaining <= 0:
                    break
                img_path, _ = base_dataset.samples[ds_idx]
                with Image.open(img_path) as pil_img:
                    manual = imagenet_transform(pil_img.convert("RGB"))
                if not torch.allclose(images[local_i], manual, atol=1e-5, rtol=1e-4):
                    max_diff = (images[local_i] - manual).abs().max().item()
                    raise AssertionError(f"Preprocessing mismatch for {img_path}; max abs diff={max_diff}")
                checks_remaining -= 1

        outputs = session.run(None, {input_name: images.contiguous().numpy()})[0]
        pred_ids = np.argmax(outputs, axis=1)

        # True class from folder name, e.g. /.../00023/img.JPEG -> 23
        true_numeric = []
        for ds_idx in batch_indices:
            img_path, _ = base_dataset.samples[ds_idx]
            folder_name = os.path.basename(os.path.dirname(img_path))
            true_numeric.append(int(folder_name))

        y_true.extend(true_numeric)
        y_pred.extend(pred_ids.tolist())

    accuracy = accuracy_score(y_true, y_pred)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )

    print(f"--- {mode} Classification Metrics ---")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision_macro:.4f} (macro)")
    print(f"Recall:    {recall_macro:.4f} (macro)")
    print(f"F1 Score:  {f1_macro:.4f} (macro)")

    return {
        'accuracy': accuracy,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
    }


metrics_fp32 = measure_classification_metrics_trt(MODEL_FILE, benchmark_loader, mode="FP32")
metrics_fp16 = measure_classification_metrics_trt(MODEL_FILE, benchmark_loader, mode="FP16")
metrics_int8 = measure_classification_metrics_trt(MODEL_FILE, benchmark_loader, mode="INT8")

## Visualize Feature Maps

We will visualize the feature maps for a few sample images across the three models.

In [ ]:



def _create_trt_session_for_mode(model_path, mode):
    use_fp16 = mode in ("FP16", "INT8")
    use_int8 = mode == "INT8"

    trt_options = {
        'device_id': 0,
        'trt_fp16_enable': use_fp16,
        'trt_int8_enable': use_int8,
        'trt_engine_cache_enable': True,
        'trt_engine_cache_path': './trt_cache',
        'trt_timing_cache_enable': True,
        'trt_timing_cache_path': './trt_cache',
    }
    if use_int8:
        trt_options.update({
            'trt_int8_use_native_calibration_table': False,
            'trt_int8_calibration_table_name': 'calibration.flatbuffers',
        })

    sess_options = ort.SessionOptions()
    sess_options.add_session_config_entry('session.dynamic_block_base', '4')

    return ort.InferenceSession(
        model_path,
        providers=['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider'],
        provider_options=[trt_options, {}, {}],
        sess_options=sess_options,
    )


def _unnormalize_for_display(img_chw):
    mean = torch.tensor([0.485, 0.456, 0.406], dtype=img_chw.dtype).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], dtype=img_chw.dtype).view(3, 1, 1)
    return (img_chw * std + mean).clamp(0.0, 1.0)


def _shape_from_value_info(vi):
    dims = []
    for d in vi.type.tensor_type.shape.dim:
        if d.HasField("dim_value"):
            dims.append(int(d.dim_value))
        elif d.HasField("dim_param") and d.dim_param:
            dims.append(d.dim_param)
        else:
            dims.append("?")
    return tuple(dims)


def _create_hooked_model_for_layer(src_model_path, target_layer_name):
    model = onnx.load(src_model_path, load_external_data=True)
    graph_output_names = {o.name for o in model.graph.output}

    if target_layer_name in graph_output_names:
        for o in model.graph.output:
            if o.name == target_layer_name:
                return src_model_path, False, _shape_from_value_info(o)
        return src_model_path, False, None

    known_names = set(v.name for v in model.graph.value_info)
    known_names.update(v.name for v in model.graph.input)
    known_names.update(v.name for v in model.graph.output)
    for node in model.graph.node:
        known_names.update(name for name in node.output if name)

    if target_layer_name not in known_names:
        raise ValueError(f"Layer '{target_layer_name}' not found in model graph.")

    inferred = onnx.shape_inference.infer_shapes(model)
    all_vi = list(inferred.graph.value_info) + list(inferred.graph.output) + list(inferred.graph.input)
    matched_vi = next((deepcopy(vi) for vi in all_vi if vi.name == target_layer_name), None)
    if matched_vi is None:
        raise ValueError(f"Could not infer shape/type for layer '{target_layer_name}'.")

    model.graph.output.append(matched_vi)
    with tempfile.NamedTemporaryFile(suffix='.onnx', delete=False) as tmpf:
        hooked_model_path = tmpf.name
    onnx.save(model, hooked_model_path)
    return hooked_model_path, True, _shape_from_value_info(matched_vi)


def prepare_layer_debugger(model_path, layer_name):
    hooked_model_path, is_temp_model, layer_shape = _create_hooked_model_for_layer(model_path, layer_name)
    sessions = {m: _create_trt_session_for_mode(hooked_model_path, m) for m in ["FP32", "FP16", "INT8"]}
    input_names = {m: sessions[m].get_inputs()[0].name for m in sessions}

    print(f"Layer: {layer_name}")
    print(f"Selected layer shape: {layer_shape}")
    print(f"Created sessions: {list(sessions.keys())}")

    return {
        "layer_name": layer_name,
        "hooked_model_path": hooked_model_path,
        "is_temp_model": is_temp_model,
        "layer_shape": layer_shape,
        "sessions": sessions,
        "input_names": input_names,
    }


In [ ]:
def compare_feature_maps(outputs, image_path, filter_index, img_tensor, layer_name, layer_shape=None):
    fp32_map, fp16_map, int8_map = outputs["FP32"], outputs["FP16"], outputs["INT8"]

    if fp32_map.ndim != 4:
        raise ValueError(f"Expected 4D feature maps, got shape {fp32_map.shape}")

    filt = int(filter_index)
    num_filters = fp32_map.shape[1]
    if filt < 0 or filt >= num_filters:
        raise ValueError(f"filter_index must be in [0, {num_filters - 1}], got {filt}")

    d_fp32_fp16 = np.abs(fp32_map[0, filt] - fp16_map[0, filt])
    d_fp16_int8 = np.abs(fp16_map[0, filt] - int8_map[0, filt])
    d_fp32_int8 = np.abs(fp32_map[0, filt] - int8_map[0, filt])
    diff_vmax = max(float(d_fp32_fp16.max()), float(d_fp16_int8.max()), float(d_fp32_int8.max())) or 1e-12

    label = os.path.normpath(str(image_path))
    if 'IMAGENET_ROOT' in globals():
        try:
            label = os.path.relpath(label, IMAGENET_ROOT)
        except Exception:
            pass

    if layer_shape is None:
        layer_shape = tuple(fp32_map.shape)

    fig, ax = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=False)
    fig.subplots_adjust(wspace=-0.3, hspace=0.1)
    fig.suptitle(
        f"Layer: {layer_name} {tuple(layer_shape)} | Image: {label} | Filter: {filt}",
        fontsize=16,
        fontweight='bold',
        y=0.95,
        x=0.53
    )

    im00 = ax[0, 0].imshow(fp32_map[0, filt], cmap='viridis')
    ax[0, 0].set_title("FP32")

    im01 = ax[0, 1].imshow(fp16_map[0, filt], cmap='viridis')
    ax[0, 1].set_title("FP16")

    im02 = ax[0, 2].imshow(int8_map[0, filt], cmap='viridis')
    ax[0, 2].set_title("INT8")

    im10 = ax[1, 0].imshow(d_fp32_fp16, cmap='plasma', vmin=0.0, vmax=diff_vmax)
    ax[1, 0].set_title("|FP32 - FP16|")

    im11 = ax[1, 1].imshow(d_fp32_int8, cmap='plasma', vmin=0.0, vmax=diff_vmax)
    ax[1, 1].set_title("|FP32 - INT8|")

    im12 = ax[1, 2].imshow(d_fp16_int8, cmap='plasma', vmin=0.0, vmax=diff_vmax)
    ax[1, 2].set_title("|FP16 - INT8|")

    cax_top = fig.add_axes([0.8625, 0.515, 0.015, 0.3625])
    fig.colorbar(im02, cax=cax_top)

    cax_bottom = fig.add_axes([0.8625, 0.1125, 0.015, 0.3625])
    fig.colorbar(im12, cax=cax_bottom)

    for r in range(2):
        for c in range(3):
            ax[r, c].axis('off')
    plt.show()


In [ ]:
def plot_layer_filter(debugger, image_path, filter_index):
    layer_name = debugger["layer_name"]
    layer_shape = debugger.get("layer_shape")
    sessions = debugger["sessions"]
    input_names = debugger["input_names"]

    pil_img = Image.open(image_path).convert("RGB")
    img_tensor = imagenet_transform(pil_img)
    if img_tensor.shape != (3, IMG_DIM, IMG_DIM):
        raise ValueError(f"Expected CHW=(3,{IMG_DIM},{IMG_DIM}), got {tuple(img_tensor.shape)}")

    input_np = img_tensor.unsqueeze(0).numpy()
    outputs = {}
    for mode in ["FP32", "FP16", "INT8"]:
        fmap = sessions[mode].run([layer_name], {input_names[mode]: input_np})[0]
        outputs[mode] = fmap.astype(np.float32)

    compare_feature_maps(outputs, image_path, filter_index, img_tensor, layer_name, layer_shape)

In [ ]:
TARGET_LAYER = "hardtanh_3"
LAYER_DEBUGGER = prepare_layer_debugger(MODEL_FILE, TARGET_LAYER)
TARGET_IMAGE_PATH = IMAGENET_ROOT + "/00029/418673857124028.jpg"
TARGET_FILTER_INDEX = 1

outputs = plot_layer_filter(LAYER_DEBUGGER, TARGET_IMAGE_PATH, TARGET_FILTER_INDEX)

In [ ]:
TARGET_LAYER = "hardtanh_7"
LAYER_DEBUGGER = prepare_layer_debugger(MODEL_FILE, TARGET_LAYER)
TARGET_IMAGE_PATH = IMAGENET_ROOT + "/00029/418673857124028.jpg"
TARGET_FILTER_INDEX = 3

outputs = plot_layer_filter(LAYER_DEBUGGER, TARGET_IMAGE_PATH, TARGET_FILTER_INDEX)

In [ ]:
TARGET_LAYER = "hardtanh_11"
LAYER_DEBUGGER = prepare_layer_debugger(MODEL_FILE, TARGET_LAYER)
TARGET_IMAGE_PATH = IMAGENET_ROOT + "/00029/418673857124028.jpg"
TARGET_FILTER_INDEX = 30

outputs = plot_layer_filter(LAYER_DEBUGGER, TARGET_IMAGE_PATH, TARGET_FILTER_INDEX)

In [ ]:
TARGET_LAYER = "hardtanh_15"
LAYER_DEBUGGER = prepare_layer_debugger(MODEL_FILE, TARGET_LAYER)
TARGET_IMAGE_PATH = IMAGENET_ROOT + "/00029/418673857124028.jpg"
TARGET_FILTER_INDEX = 130

outputs = plot_layer_filter(LAYER_DEBUGGER, TARGET_IMAGE_PATH, TARGET_FILTER_INDEX)

In [ ]:
TARGET_LAYER = "hardtanh_34"
LAYER_DEBUGGER = prepare_layer_debugger(MODEL_FILE, TARGET_LAYER)


TARGET_IMAGE_PATH = IMAGENET_ROOT + "/00029/418673857124028.jpg"
TARGET_FILTER_INDEX = 56

outputs = plot_layer_filter(LAYER_DEBUGGER, TARGET_IMAGE_PATH, TARGET_FILTER_INDEX)
